# Lecture 2: Signal and noise — from single genes to signatures

**Guiding idea.** Bulk RNA-seq, qPCR and in situ hybridisation average over millions of cells, so every gene
is read through many molecules and its relative sampling noise (≈ 1/√λ, with λ the expected count) is
negligible. That is why **one marker gene** (*PAX6*, *EOMES*, *TBR1*…) used to be enough, and why the
bulk benchmarking of organoids against BrainSpan (PCA, correlation, WGCNA) works.

Single-cell and Visium HD measure **one cell or one bin**: a few hundred to a few thousand molecules spread
over ~20,000 genes. For most genes λ < 1, so **zero is the most likely reading even when the gene is
expressed**: P(0) = e^(−λ). At this resolution a single gene is no longer a reliable readout, and information
must be pooled:

- **across genes**: signatures and multi-gene scores, modules (WGCNA), PCA — averaging *k* genes improves
  signal-to-noise roughly as √k;
- **across space**: larger bins, smoothing over neighbouring bins;
- **across cells**: clusters and pseudobulk.

A note on words: in droplet UMI data most zeros are *not* a technical "dropout" artefact but plain Poisson /
negative binomial sampling at low depth (Svensson 2020). The problem is that counts are few, not that the
technology is broken. Exercise 3 checks this on real data.

Same way of working as in Lecture 1: read the question, complete the **decomposition tree**, write
**one function per leaf** and a root function that calls them.

**Datasets** (in `Datasets/`):
- `e18_mouse_brain_*` — 1,301 cells from embryonic day 18 mouse cortex, hippocampus and ventricular zone
  (10x Genomics public dataset *1k Brain Cells from an E18 Mouse*, Chromium v3). Sparse count matrix
  cells × genes (`scipy.sparse`) + gene names.
- `visiumhd_mouse_brain_*` — crops of a 10x Genomics Visium HD section of the adult mouse brain
  (public dataset *Visium HD Mouse Brain, fresh frozen*), at 2, 8 and 16 µm bins, for a panel of marker genes.
  For each resolution: a sparse count matrix bins × genes, and a table of bins with grid position and total UMIs
  (over **all** genes, not only the panel).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, ndimage, sparse
from scipy.cluster.vq import kmeans2

rng = np.random.default_rng(2026)

sc_counts = sparse.load_npz("../Datasets/e18_mouse_brain_counts.npz").tocsr()   # cells × genes
sc_genes = pd.read_csv("../Datasets/e18_mouse_brain_genes.csv")["gene"].to_numpy()
print("single-cell:", sc_counts.shape)

## Exercise 1 — How many molecules do you need to see a gene? (simulation)

**Question:** take the same expression profile (the fraction of mRNA molecules that comes from each gene)
and read it at the depth of a bulk library, of a good single cell, of a poor single cell and of a Visium HD
bin. What happens to each gene?

The expected count of gene *g* is λ_g = depth × fraction_g, and the observed count is Poisson(λ_g).

```
From bulk to a single bin
├── Profile          → make_profile(n_genes): log-normal fractions that sum to 1
├── Read it          → read_out(profile, depth, n_obs): Poisson counts, n_obs observations
├── Detection        → detection_rate(counts): fraction of observations with count > 0, per gene
├── Precision        → cv(counts): standard deviation / mean, per gene
└── Plot             → detection and CV vs λ, one curve per depth, with the theory 1 − e^(−λ) and 1/√λ
```

**To do:** use depths 20,000,000 (bulk), 10,000 (good cell), 2,000 (poor cell), 150 (8 µm Visium HD bin)
and 10 (2 µm bin): the last two are the median total UMIs per bin of the Visium HD section of Exercise 4. For each depth, report how many genes have λ ≥ 10 (≈ a CV below 30%).

*Discussion question:* at which depth would you still trust "gene X is not expressed in this cell"?

In [ ]:
DEPTHS = {"bulk": 20_000_000, "good cell": 10_000, "poor cell": 2_000, "8 µm bin": 150, "2 µm bin": 10}

def make_profile(n_genes=15_000, rng=rng):
    # TODO: log-normal weights normalised to sum to 1
    pass

def read_out(profile, depth, n_obs=300, rng=rng):
    # TODO
    pass

def detection_rate(counts):
    # TODO
    pass

def cv(counts):
    # TODO
    pass

def summarise_depths(profile):
    # TODO: one row per depth: median λ, genes with λ ≥ 1, genes with λ ≥ 10
    pass

def plot_detection_and_cv(profile, n_genes_plot=1500):
    # TODO
    pass

## Exercise 2 — One marker or a signature?

### 2a — Simulation

**Question:** two populations (say radial glia and neurons) differ by a factor 3 in 50 marker genes.
How well can we tell them apart using 1, 2, 5, 10, 20 or 50 of the markers, at different depths?

We measure separability with the **AUC**: the probability that a random cell of population B has a higher
score than a random cell of population A (0.5 = coin toss, 1 = perfect). It is the Mann–Whitney U statistic
divided by n_A × n_B.

```
Marker vs signature
├── Simulate         → simulate_two_populations(depth, n_markers=50, fold=3): counts for A and B
├── Normalise        → log1p(counts / total × 10,000)
├── Score            → signature_score(norm, markers): mean of the normalised marker genes
├── Separability     → auc(score_a, score_b)
└── Grid             → AUC for k ∈ {1, 2, 5, 10, 20, 50} × depth ∈ {200, 1,000, 5,000, 20,000}
```

*Discussion question:* the gain from adding markers is large at low depth and small at high depth. Why?
What would happen if half of your "markers" were not really markers?

In [ ]:
def simulate_two_populations(depth, n_markers=50, n_background=2000, fold=3.0, n_cells=300, rng=rng):
    # TODO: the first n_markers genes are 'fold' times higher in B; Poisson counts at the given depth
    pass

def normalise(counts, target=10_000):
    # TODO
    pass

def signature_score(norm, markers):
    # TODO
    pass

def auc(score_a, score_b):
    # TODO: hint, stats.mannwhitneyu(...).statistic / (n_a * n_b)
    pass

def auc_grid(ks=(1, 2, 5, 10, 20, 50), depths=(200, 1_000, 5_000, 20_000)):
    # TODO
    pass

### 2b — Real data: does a progenitor have *Pax6*?

In the E18 mouse brain cells we score each cell with a **radial glia / progenitor signature** and look
at how often the classic single marker *Pax6* is zero in cells that the signature calls progenitors.

```
Single marker vs signature on real cells
├── Normalise        → log1p(counts / total × 10,000)
├── Signature score  → mean of the progenitor genes (not including the single markers we test)
├── Define groups    → top 20% of the score = "progenitor-like", bottom 50% = "not progenitor"
└── Compare          → fraction of Pax6 = 0 in each group; same for the neuronal marker Neurod6 and a neuronal signature
```

*Discussion question:* if you sorted cells by "*Pax6* > 0", what fraction of the progenitor-like cells
would you lose? Is the signature "true"? How would you check it does not just reflect depth?

In [ ]:
# the single markers we test (Pax6, Sox2, Neurod6, Tbr1) are NOT in the signatures, to avoid circularity
PROGENITOR = ["Hes1", "Hes5", "Vim", "Nes", "Fabp7", "Slc1a3", "Mki67", "Top2a", "Ccnd2", "Notch1", "Fgfr3", "Sox9"]
NEURON = ["Neurod2", "Bcl11b", "Satb2", "Stmn2", "Tubb3", "Mef2c", "Gap43", "Syt1", "Nrxn1", "Dcx"]

def gene_index(names):
    # TODO: positions of the genes in sc_genes (warn about missing ones)
    pass

def normalise_sparse(counts, target=10_000):
    # TODO: hint, counts.multiply(...).log1p() keeps the matrix sparse
    pass

def score(norm, genes):
    # TODO
    pass

def zero_fraction_by_group(counts, gene, score_values):
    # TODO
    pass

## Exercise 3 — Zeros in real data: dropout or sampling?

**Question:** are there *more* zeros in the E18 data than expected from simple sampling? If each gene *g*
has a fraction μ_g of the molecules and cell *i* has total *s_i*, the Poisson model predicts

$$P(\text{zero}_g) = \frac{1}{N}\sum_i e^{-\mu_g s_i}$$

and a negative binomial with overdispersion φ (biological variability between cells) predicts
$\frac{1}{N}\sum_i (1+\varphi\,\mu_g s_i)^{-1/\varphi}$.

```
Zero inflation check
├── Per gene         → mean count, observed fraction of zeros
├── Poisson          → expected_zeros_poisson(mu, totals)
├── Neg. binomial    → expected_zeros_nb(mu, totals, phi) for phi in {0.1, 0.5}
└── Plot             → observed vs expected zero fraction against mean count (log x)
```

*Discussion question:* the cells are a mixture of different cell types. Why does this alone produce
"extra" zeros compared to the Poisson line? So, is a gene that is zero in a cell "dropped out"?

In [ ]:
def gene_stats(counts):
    # TODO: per-gene mean, fraction of zeros, mu = share of all molecules; also return cell totals
    pass

def expected_zeros_poisson(mu, totals):
    # TODO
    pass

def expected_zeros_nb(mu, totals, phi):
    # TODO
    pass

def plot_zeros(df):
    # TODO
    pass

## Exercise 4 — Visium HD: resolution against signal

The Visium HD file `visiumhd_mouse_brain_cortex_2um.npz` is a 1.7 × 1.7 mm crop of an adult mouse brain
section at the native **2 µm** resolution: from the striatum (left) through the white matter to the cortex and
the pia (right). It contains 67 marker genes and, for every bin, the total UMIs over **all** ~19,000 genes.
`visiumhd_mouse_brain_gene_means.csv` gives, for every gene, its mean count per bin over the whole section
at 2, 8 and 16 µm.

The 8 and 16 µm bins of the 10x pipeline are simply sums of 4×4 and 8×8 bins of 2 µm: we can rebuild them
ourselves.

### 4a — How many molecules per bin?

```
Counts per bin at each resolution
├── Load            → load_visium(path): sparse counts (bins × genes), bins table (row, col, total_counts), genes
├── Aggregate       → aggregate(counts, bins, size_um): sum b×b neighbouring 2 µm bins (b = size / 2)
├── Summary         → summarise_resolution(...): bins, median total UMIs, median and % zero for a few genes
└── Whole section   → from the gene-means table: genes with mean ≥ 1 and ≥ 10 per bin, per resolution
```

Compare with Exercise 1: which line of that table does a Visium HD bin correspond to?

In [ ]:
def load_visium(path):
    z = np.load(path)
    counts = sparse.csr_matrix((z["data"], z["indices"], z["indptr"]), shape=tuple(z["shape"]))
    bins = pd.DataFrame({"row": z["row"].astype(int), "col": z["col"].astype(int),
                         "total_counts": z["total_counts"].astype(float)})
    return counts, bins, list(z["genes"])

def aggregate(counts, bins, size_um):
    # TODO: give every 2 µm bin the key (row // b, col // b), then sum counts and totals with the same key
    # hint: a sparse "assignment" matrix (new bins × old bins) times counts does the sum in one step
    pass

def summarise_resolution(counts, bins, genes, size_um, example_genes=("Slc17a7", "Mbp", "Rorb")):
    # TODO
    pass

vh_counts, vh_bins, vh_genes = load_visium("../Datasets/visiumhd_mouse_brain_cortex_2um.npz")
gene_means = pd.read_csv("../Datasets/visiumhd_mouse_brain_gene_means.csv")

### 4b — One gene or a signature, in space

The layer-4/5 marker *Rorb* should draw a band parallel to the pia. Compare it with a **layer 4/5
signature** of 5 genes, at 8 and 16 µm, and with the single gene smoothed over neighbouring bins.

```
Maps of one gene vs a signature
├── Score per bin      → bin_score(counts, bins, genes): log1p(1000 × sum of the genes / total UMIs)
├── To an image        → to_image(values, bins): 2D array, NaN where there is no tissue
├── Smooth             → Gaussian filter on the single-gene image (σ = 2 bins)
└── Grid of maps       → rows: 8 µm, 16 µm; columns: Rorb, Rorb smoothed, L4/5 signature
```

### 4c — How reproducible is a bin?

We cannot measure the same bin twice, but with **count splitting** (each molecule goes to half A or half B
with probability 0.5, see also Exercise 6b) we get two independent half-measurements of every bin.
The correlation between the two halves across bins says how much of what we see is signal.

```
Split-half reliability
├── Split          → split_counts(counts): Binomial(counts, 0.5) and the rest (totals split the same way)
├── Score halves   → bin_score on each half
├── Correlate      → Pearson r between half A and half B across bins
└── Table          → r for single genes and for signatures, at 2, 4, 8, 16 and 32 µm
```

*Discussion question:* how large must a bin be before *Rorb* alone is reproducible? Is that still
"single-cell resolution"? Does the signature need the same bin size?

In [ ]:
SIGNATURES = {
    "L2/3 IT":          ["Cux2", "Lamp5", "Otof", "Calb1", "Stard8", "Rasgrf2"],
    "L4/5 IT":          ["Rorb", "Rspo1", "Whrn", "Scnn1a", "Kcnh5"],
    "L5 ET":            ["Fezf2", "Bcl11b", "Pou3f1", "Parm1"],
    "L6":               ["Foxp2", "Syt6", "Tle4", "Crym", "Nxph4", "Ccn2"],
    "Oligodendrocytes": ["Mbp", "Plp1", "Mobp", "Mog", "Mag", "Cnp", "Cldn11"],
    "Striatal MSN":     ["Ppp1r1b", "Penk", "Drd1", "Drd2", "Tac1", "Adora2a", "Rgs9", "Gpr88"],
    "Astrocytes":       ["Gfap", "Aqp4", "Aldh1l1", "Slc1a3", "Gja1", "Slc1a2"],
    "Inhibitory":       ["Gad1", "Gad2", "Slc32a1", "Pvalb", "Sst", "Vip"],
    "Microglia":        ["P2ry12", "Csf1r", "Hexb", "C1qa", "Tmem119"],
    "Endothelial":      ["Cldn5", "Flt1", "Pecam1"],
}

def bin_score(counts, bins, genes):
    # TODO: log1p(1000 × summed counts of the genes / total UMIs of the bin)
    pass

def to_image(values, bins):
    # TODO
    pass

def smooth_image(image, sigma=2):
    # TODO: Gaussian filter that ignores the NaN (no tissue) pixels
    pass

def plot_maps(sizes=(8, 16), gene="Rorb", signature="L4/5 IT"):
    # TODO
    pass

In [ ]:
def split_counts(counts, bins, rng=rng):
    # TODO: binomial thinning of the counts AND of the totals; return ((counts_a, bins_a), (counts_b, bins_b))
    pass

def split_half_reliability(counts, bins, genes):
    # TODO
    pass

def reliability_table(features, sizes=(2, 4, 8, 16, 32)):
    # TODO
    pass

## Exercise 5 — Populations come from single-cell data

The cell types of a spatial section are not discovered in the spatial data: they come from **single-cell
atlases** (for the mouse brain: Zeisel et al. 2018, Tasic et al. 2018, Yao et al. 2021), and the spatial data are
annotated by transferring them. The lists in `SIGNATURES` are marker genes of the atlas cell types.
Real tools (RCTD, cell2location, Tangram) use the full single-cell profiles; here we use the simplest version:
signature scores.

```
Label transfer by signature scores
├── Score matrix      → score_matrix(counts, bins): one column per signature (bin_score), z-scored across bins
├── Label             → the signature with the highest z-score
├── Confidence        → margin = best z − second best z; "ambiguous" if margin < 0.5
├── Maps              → label map at 8 µm, 8 µm with scores smoothed over neighbours, 16 µm and 32 µm
└── Summary           → % ambiguous bins and label composition per resolution
```

*Discussion question:* small bins are ambiguous because they are noisy; big bins are ambiguous because
they contain several cells. Is there a "right" bin size? Why do the spatial data still need the single-cell
reference, and what happens to a cell type that is missing from it (for example, a cell type that exists only
in organoids)?

In [ ]:
def score_matrix(counts, bins):
    # TODO: one column per signature, z-scored across bins
    pass

def label_bins(z, min_margin=0.5):
    # TODO: return (labels, margin)
    pass

def smooth_scores(z, bins, sigma):
    # TODO: smooth each score column as an image (smooth_image), then read the values back at the bins
    pass

def plot_labels(panels=((8, 0), (8, 3), (16, 0), (32, 0))):
    # TODO: one label map per (bin size, smoothing sigma) + table of % of bins per label
    pass

## Exercise 6 — Analysis traps

| Part | Data | Trap |
|---|---|---|
| 6a | E18 cells | sequencing depth as a confounder, and what normalisation does (not) fix |
| 6b | simulation | double dipping: clustering and testing on the same data |
| 6c | simulation | pseudoreplication: cells are not biological replicates |
| 6d | Visium HD | spillover: a marker where its cells are not |

### 6a — Deeper cells "express more" of everything

**Question:** take the E18 cells, pick a random half and **thin** their counts to 30% (each molecule kept with
probability 0.3). Same cells, same biology: only the sequencing depth differs. Then test the 2,000 most
expressed genes between the two halves, with three versions of the data:

```
Depth as a confounder
├── Thin          → thin_cells(counts, fraction=0.3): binomial thinning of a random half of the cells
├── Versions      → raw counts · counts / total (proportions) · log1p(counts / total × 10,000)
├── Test          → t-test per gene, fraction of genes with p < 0.01 (should be ≈ 1%)
└── Real data     → the same test on proportions, splitting the real cells by depth (above / below the median)
```

*Discussion question:* why does the log1p version still "see" depth, although it is divided by the total?
(hint: E[log(X)] ≠ log(E[X]) when X is often 0 — Lun 2018.) And in the real split, why are so many genes still
different after normalisation: what else differs between deep and shallow cells?

In [ ]:
def thin_cells(counts, fraction=0.3, rng=rng):
    # TODO: pick a random half of the cells and replace their counts with Binomial(count, fraction)
    pass

def versions(counts, n_genes=2000):
    # TODO: dict with raw counts, counts / total, log1p(counts / total × 10⁴) for the top n_genes
    pass

def fraction_significant_genes(matrix, group, alpha=0.01):
    # TODO
    pass

def depth_confounding(counts):
    # TODO
    pass

### 6b — Double dipping

**Question:** simulate cells that are **all the same** (pure Poisson noise, no populations), cluster them
into 2 clusters, then test every gene between the two clusters. How many genes are "differentially
expressed"?

The fix shown here is **count splitting** (Neufeld et al. 2024): for Poisson data, `train = Binomial(X, 0.5)`
and `test = X − train` are independent. Cluster on `train`, test on `test`.

```
Double dipping
├── Simulate      → 500 identical cells × 200 genes, Poisson
├── Cluster       → log1p, PCA (5 components), k-means with 2 clusters
├── Test          → t-test of every gene between clusters, on the same data
├── Split         → count splitting: cluster on train, test on test
└── Compare       → fraction of genes with p < 0.05 (should be ≈ 5%)
```

*Discussion question:* in a real analysis, the clusters you test are the ones you found because they
differed. What does a p-value mean then? When is it still useful to look at "markers" of clusters?

In [ ]:
def simulate_identical_cells(n_cells=500, n_genes=200, rng=rng):
    # TODO
    pass

def cluster(counts, n_pcs=5, seed=0):
    # TODO: log1p, centre, SVD, kmeans2 on the first PCs
    pass

def fraction_significant(counts, labels, alpha=0.05):
    # TODO
    pass

def count_split(counts, rng=rng):
    # TODO
    pass

def double_dipping():
    # TODO
    pass

### 6c — Pseudoreplication: cells are not replicates

**Question:** 3 control and 3 treated organoids, 300 cells each. The treatment does **nothing**, but every
organoid has its own expression level of the gene (organoid-to-organoid variability). Test with
n = cells, then with n = organoids (**pseudobulk**: sum the counts of each organoid).

```
Pseudoreplication
├── Simulate       → simulate_experiment(): organoid level ~ log-normal(sd 0.3), cells Poisson around it
├── Cell-level     → Mann–Whitney on 900 vs 900 cells
├── Pseudobulk     → sum per organoid, divide by number of cells, t-test on log values, 3 vs 3
└── Repeat         → false positive rate over 300 experiments
```

*Discussion question:* in organoid studies, what is the replicate: the organoid, the differentiation
batch, or the iPSC line? What does Squair et al. (2021) recommend?

In [ ]:
def simulate_experiment(n_organoids=3, n_cells=300, mean=2.0, organoid_sd=0.3, rng=rng):
    # TODO: one row per cell with condition, organoid and count
    pass

def cell_level_test(df):
    # TODO
    pass

def pseudobulk(df):
    # TODO
    pass

def pseudobulk_test(df):
    # TODO
    pass

def false_positive_rates(n_experiments=300):
    # TODO
    pass

### 6d — A marker where its cells are not (Visium HD)

**Question:** *Ppp1r1b* (DARPP-32) marks the medium spiny neurons of the striatum, on the left of the crop.
How far into the white matter and cortex does its signal reach? Compare with the oligodendrocyte marker
*Mbp*, which marks the white matter.

```
Lateral spillover
├── Aggregate      → 16 µm bins
├── Profile        → for each column of bins (left → right), mean normalised expression over all rows
├── Plot           → profiles of Ppp1r1b and Mbp vs distance (µm), log scale
└── Read           → where does Ppp1r1b fall to its cortical background level?
```

*Discussion question:* part of the signal outside the striatum is real (some cortical neurons express
*Ppp1r1b*), part is mRNA that diffused laterally during permeabilisation, or ambient RNA. How would you
tell the two apart? What does this mean for the "single-cell resolution" of a 2 µm grid?

In [ ]:
def column_profile(counts, bins, gene):
    # TODO
    pass

def plot_profiles(genes=("Ppp1r1b", "Mbp", "Slc17a7"), size=16):
    # TODO
    pass

## Wrap-up

Fill in the table: for bulk, single cell and a Visium HD bin, write the order of magnitude of molecules per
observation, whether a single marker gene is reliable, and what you would do instead.